## 1. Imports

In [1]:
import chromadb
import ollama
import numpy as np

print("✅ Librerías cargadas")

✅ Librerías cargadas


## 2. Configuración

In [2]:
# Modelos (los que creaste con 'ollama create')
MODELO_LLM = "gemma4:latest"
MODELO_EMBEDDING = "bge-m3:latest"

# Parámetros del RAG
CHUNK_SIZE = 600              # tamaño objetivo de cada chunk en caracteres
N_CHUNKS_RECUPERADOS = 5      # cuántos chunks pasamos al LLM como contexto

print(f"Configuración:")
print(f"  LLM:                {MODELO_LLM}")
print(f"  Embeddings:         {MODELO_EMBEDDING}")
print(f"  Chunk size:         {CHUNK_SIZE} chars")
print(f"  Chunks por consulta:{N_CHUNKS_RECUPERADOS}")

Configuración:
  LLM:                gemma4:latest
  Embeddings:         bge-m3:latest
  Chunk size:         600 chars
  Chunks por consulta:5


## 3. Documentos de TechShop

Base de conocimiento del asistente. Son 4 textos con las políticas oficiales de la empresa.

In [3]:
# ============================================================
# DOCUMENTOS DE TECHSHOP - Base de conocimiento
# ============================================================

politica_devoluciones = """
POLÍTICA DE DEVOLUCIONES Y REEMBOLSOS - TechShop

Plazo de devolución:
Los clientes tienen 30 días naturales desde la fecha de recepción del pedido para solicitar 
una devolución. Pasado este plazo, no se aceptarán devoluciones salvo defectos de fabricación.

Condiciones para la devolución:
- El producto debe estar en su estado original, sin uso y con todos sus accesorios
- Debe conservarse el embalaje original en buen estado
- Se debe incluir la factura de compra o número de pedido
- Los productos personalizados o con el precinto roto no son devolvibles (excepto defectos)

Proceso de devolución:
1. Contacta con atención al cliente en devoluciones@techshop.es o llamando al 900 123 456
2. Recibirás un número de autorización de devolución (RMA) en 24-48 horas
3. Envía el producto con el RMA visible en el exterior del paquete
4. Una vez recibido e inspeccionado el producto, procesamos el reembolso en 5-7 días hábiles

Gastos de envío en devoluciones:
- Si el producto tiene defecto de fábrica: TechShop asume los gastos de envío
- Si la devolución es por cambio de opinión: el cliente asume los gastos de envío
- El coste del envío de devolución no se reembolsa salvo defectos

Reembolso:
El reembolso se realiza por el mismo método de pago utilizado en la compra. 
Las tarjetas de crédito pueden tardar hasta 10 días adicionales según el banco.
"""

politica_envios = """
POLÍTICA DE ENVÍOS - TechShop

Plazos de entrega:
- Envío estándar (GRATIS en pedidos +50€): 3-5 días hábiles
- Envío express (4,99€): 1-2 días hábiles
- Envío urgente mismo día (9,99€, solo Madrid y Barcelona): disponible en pedidos antes de las 12:00h
- Envíos internacionales (zona UE): 7-14 días hábiles, precio según destino

Seguimiento del pedido:
Recibirás un email con el número de seguimiento cuando tu pedido sea enviado. 
Puedes rastrear tu pedido en la web de la empresa de transportes o en Mi Cuenta en nuestra web.

Incidencias con el envío:
- Paquete dañado: fotografía los daños antes de abrir e infórmanos en 48 horas
- Paquete perdido: si pasan más de 10 días hábiles sin recibir el pedido, contacta con nosotros
- Dirección incorrecta: contacta inmediatamente. Si el paquete ya salió, pueden aplicarse cargos adicionales

Empresas de transporte colaboradoras:
Trabajamos con SEUR, MRW y Correos Express. La elección depende de tu zona y el tipo de envío.

Restricciones:
No realizamos envíos a apartados de correos. Para Canarias, Ceuta y Melilla 
pueden aplicarse tasas aduaneras adicionales que son responsabilidad del cliente.
"""

garantias_productos = """
GARANTÍAS DE PRODUCTOS - TechShop

Garantía legal:
Todos los productos vendidos en TechShop tienen garantía legal de 2 años según la normativa europea 
(Directiva 2019/771/UE). Esta garantía cubre defectos de fabricación y materiales.

Garantía del fabricante:
Además de la garantía legal, muchos productos incluyen garantía adicional del fabricante:
- Portátiles y ordenadores: generalmente 1-2 años adicionales según marca
- Smartphones: 1 año adicional en marcas como Apple, Samsung, Xiaomi
- Periféricos y accesorios: 1 año según fabricante
- Televisores: hasta 5 años de garantía en marcas premium (LG, Sony, Samsung)

Qué cubre la garantía:
✓ Defectos de fabricación
✓ Fallos de hardware no causados por el usuario
✓ Problemas de pantalla (píxeles muertos según umbrales del fabricante)

Qué NO cubre la garantía:
✗ Daños por golpes, caídas o líquidos
✗ Desgaste normal por uso
✗ Daños causados por uso inadecuado
✗ Accesorios consumibles (baterías, fundas, cables desgastados)
✗ Daños estéticos sin afectar al funcionamiento

Cómo reclamar la garantía:
1. Contacta con garantias@techshop.es con tu número de pedido y descripción del problema
2. Nuestro equipo técnico evaluará el caso en 48 horas
3. Si procede, te enviaremos instrucciones para enviar el producto (gastos pagados por TechShop)
4. Tiempo de resolución: 10-15 días hábiles
5. Solución: reparación, sustitución o reembolso según disponibilidad
"""

formas_pago = """
MÉTODOS DE PAGO ACEPTADOS - TechShop

Tarjetas de crédito y débito:
Aceptamos Visa, Mastercard y American Express. El pago es seguro y cifrado mediante SSL.
Para pedidos superiores a 300€, puede solicitarse verificación adicional por seguridad.

PayPal:
Pago inmediato y seguro. También puedes usar el saldo de tu cuenta PayPal o vincular 
tu tarjeta. TechShop no accede a los datos bancarios del cliente.

Bizum:
Disponible para pagos hasta 2.500€ al día. Solo para clientes con banco español.

Transferencia bancaria:
Solo para pedidos empresariales (B2B) superiores a 500€. El pedido se procesa 
cuando se confirma el ingreso, normalmente en 1-2 días hábiles.

Financiación:
Ofrecemos financiación sin intereses de 3 a 12 meses en pedidos superiores a 200€, 
en colaboración con Cofidis. Solo para residentes en España. Sujeto a aprobación.

Pago en tienda:
En nuestras tiendas físicas aceptamos efectivo, tarjeta y Bizum. 
No aceptamos cheques ni transferencias inmediatas en tienda.

Seguridad:
Todos los pagos online están protegidos con protocolo 3D Secure y cifrado SSL de 256 bits.
TechShop no almacena datos de tarjetas de crédito.
"""

print("📄 Documentos de TechShop cargados:")
print(f"  - Política de devoluciones: {len(politica_devoluciones)} caracteres")
print(f"  - Política de envíos: {len(politica_envios)} caracteres")
print(f"  - Garantías: {len(garantias_productos)} caracteres")
print(f"  - Formas de pago: {len(formas_pago)} caracteres")



📄 Documentos de TechShop cargados:
  - Política de devoluciones: 1358 caracteres
  - Política de envíos: 1151 caracteres
  - Garantías: 1416 caracteres
  - Formas de pago: 1143 caracteres


In [4]:
# Lista de tuplas (texto, nombre_fuente) - así indexamos todo en bucle
DOCUMENTOS = [
    (politica_devoluciones, "Política de Devoluciones"),
    (politica_envios,       "Política de Envíos"),
    (garantias_productos,   "Garantías de Productos"),
    (formas_pago,           "Métodos de Pago"),
]

print(f"📄 {len(DOCUMENTOS)} documentos cargados:")
for texto, fuente in DOCUMENTOS:
    print(f"   - {fuente}: {len(texto)} caracteres")

📄 4 documentos cargados:
   - Política de Devoluciones: 1358 caracteres
   - Política de Envíos: 1151 caracteres
   - Garantías de Productos: 1416 caracteres
   - Métodos de Pago: 1143 caracteres


## 4. Chunking semántico

Partimos cada documento respetando sus párrafos (separados por línea en blanco). Agrupamos párrafos consecutivos hasta acercarnos al `chunk_size` objetivo. Esto evita cortar frases o palabras a la mitad.

In [ ]:
def dividir_en_chunks_por_parrafos(texto, chunk_size=CHUNK_SIZE):
    """
    Divide un texto en chunks respetando los párrafos.
    
    - Los párrafos se identifican por dobles saltos de línea (\n\n).
    - Se agrupan párrafos consecutivos hasta acercarse a chunk_size.
    - Un párrafo más grande que chunk_size queda solo (no se trocea).
    """
    parrafos = [p.strip() for p in texto.split("\n\n") if p.strip()] # eliminar párrafos vacíos
    
    chunks = []
    actual = ""
    
    for parrafo in parrafos:
        # Si al añadir el siguiente párrafo nos pasamos del límite, cerramos el chunk
        if actual and len(actual) + len(parrafo) + 2 > chunk_size:
            chunks.append(actual)
            actual = parrafo
        else:
            actual = actual + "\n\n" + parrafo if actual else parrafo
    
    if actual:
        chunks.append(actual)
    
    return chunks


# Comprobación rápida con el primer documento
chunks_test = dividir_en_chunks_por_parrafos(politica_devoluciones)
print(f"Política de Devoluciones → {len(chunks_test)} chunks")
for i, c in enumerate(chunks_test, 1):
    print(f"  Chunk {i}: {len(c)} chars")

Política de Devoluciones → 3 chunks
  Chunk 1: 579 chars
  Chunk 2: 346 chars
  Chunk 3: 427 chars


## 5. Clase `AsistenteTechShop`

Encapsula todo el sistema: embedder, base vectorial, indexación, recuperación y generación.

Notas sobre la implementación:
- En `indexar_documento` anteponemos `"{fuente}. "` al texto **solo para el embedding**, no para lo que se guarda. Es un truco que mejora la recuperación.
- En `preguntar` cada chunk del contexto va etiquetado con su fuente, así el LLM sabe qué citar.

In [6]:
class AsistenteTechShop:
    """Sistema RAG para atención al cliente de TechShop."""
    
    def __init__(self, chunk_size=CHUNK_SIZE, n_chunks=N_CHUNKS_RECUPERADOS):
        self.chunk_size = chunk_size
        self.n_chunks = n_chunks
        self.modelo_llm = MODELO_LLM
        self.modelo_embedding = MODELO_EMBEDDING
        
        # ChromaDB en memoria. Si la colección existe, la reseteamos.
        self.cliente = chromadb.Client()
        try:
            self.cliente.delete_collection("techshop")
        except Exception:
            pass
        self.coleccion = self.cliente.create_collection("techshop")
        
        self.n_docs_indexados = 0
        print(f"✅ Asistente TechShop inicializado")
    
    def _embed(self, textos):
        """Genera embeddings con Ollama. Acepta string o lista de strings."""
        es_string = isinstance(textos, str)
        textos_list = [textos] if es_string else textos
        
        embeddings = []
        for t in textos_list:
            resp = ollama.embeddings(model=self.modelo_embedding, prompt=t)
            embeddings.append(resp["embedding"])
        
        arr = np.array(embeddings)
        return arr[0] if es_string else arr
    
    def indexar_documento(self, texto, fuente):
        """Trocea el documento y guarda los chunks en la BD vectorial."""
        chunks = dividir_en_chunks_por_parrafos(texto, self.chunk_size)
        
        # Truco: anteponemos la fuente al texto que se EMBEBE (no al que se guarda).
        # Esto ancla cada chunk a su temática y mejora la búsqueda.
        chunks_para_embedding = [f"{fuente}. {c}" for c in chunks]
        embeddings = self._embed(chunks_para_embedding)
        
        ids = [f"doc_{self.n_docs_indexados}_{i}" for i in range(len(chunks))]
        metadatos = [{"fuente": fuente, "chunk_idx": i} for i in range(len(chunks))]
        
        self.coleccion.add(
            ids=ids,
            embeddings=embeddings.tolist(),
            documents=chunks,            # se guarda el chunk original, sin prefijo
            metadatas=metadatos,
        )
        
        self.n_docs_indexados += 1
        print(f"✅ '{fuente}' indexado ({len(chunks)} chunks)")
    
    def _recuperar(self, pregunta):
        """Busca los n_chunks más relevantes para una pregunta."""
        emb = self._embed(pregunta).tolist()
        res = self.coleccion.query(query_embeddings=[emb], n_results=self.n_chunks)
        chunks = res["documents"][0]
        fuentes = [m["fuente"] for m in res["metadatas"][0]]
        return chunks, fuentes
    
    def preguntar(self, pregunta, mostrar_contexto=False):
        """Responde una pregunta usando RAG."""
        chunks, fuentes = self._recuperar(pregunta)
        
        # Cada chunk del contexto va etiquetado con su fuente
        bloques = [f"[Fuente: {f}]\n{c}" for c, f in zip(chunks, fuentes)]
        contexto = "\n\n---\n\n".join(bloques)
        
        if mostrar_contexto:
            print("📋 Contexto recuperado:")
            for f, c in zip(fuentes, chunks):
                print(f"   [{f}] {c[:80]}...")
            print()
        
        prompt = f"""Eres un asistente virtual de atención al cliente de TechShop, una tienda online de tecnología.
Tu objetivo es responder preguntas de clientes de forma clara, profesional y amable.

Reglas ESTRICTAS que debes cumplir SIEMPRE:
1. Responde ÚNICAMENTE con información que aparezca literalmente en el CONTEXTO de abajo.
2. NO inventes URLs, direcciones web, ni enlaces. Si no hay URL en el contexto, no des ninguna.
3. NO inventes direcciones de email. Las únicas válidas son las que aparecen literalmente en el contexto (devoluciones@techshop.es, garantias@techshop.es).
4. Copia los números (precios, plazos, días, porcentajes) EXACTAMENTE como aparecen en el contexto. No los redondees ni los confundas con otros números cercanos.
5. Si el contexto no tiene información suficiente para responder, dilo claramente y sugiere contactar con atención al cliente en devoluciones@techshop.es o al 900 123 456.
6. Al final de tu respuesta, indica entre paréntesis las fuentes oficiales en las que te has basado, usando solo los nombres entre corchetes [Fuente: ...] que veas en el contexto.
7. Sé conciso. No repitas la pregunta. No añadas información que no esté en el contexto.

CONTEXTO:
{contexto}

PREGUNTA DEL CLIENTE: {pregunta}

RESPUESTA:"""
        
        respuesta = ollama.chat(
            model=self.modelo_llm,
            messages=[{"role": "user", "content": prompt}],
        )
        return respuesta["message"]["content"]


print("Clase AsistenteTechShop definida ✅")

Clase AsistenteTechShop definida ✅


## 6. Inicialización: crear el asistente e indexar los documentos

In [7]:
asistente = AsistenteTechShop()

for texto, fuente in DOCUMENTOS:
    asistente.indexar_documento(texto, fuente)

print(f"\n🎉 Asistente listo con {asistente.n_docs_indexados} documentos")

✅ Asistente TechShop inicializado
✅ 'Política de Devoluciones' indexado (3 chunks)
✅ 'Política de Envíos' indexado (3 chunks)
✅ 'Garantías de Productos' indexado (4 chunks)
✅ 'Métodos de Pago' indexado (3 chunks)

🎉 Asistente listo con 4 documentos


## 7. Batería de pruebas

Probamos con preguntas variadas. La última es un control: una pregunta que NO está en los documentos, para verificar que el asistente no se inventa la respuesta.

In [8]:
preguntas_prueba = [
    "¿Cuánto tarda en procesarse un reembolso?",
    "¿Puedo pagar con Bizum?",
    "¿Cuánto cuesta el envío express?",
    "¿La garantía cubre si se me cae el móvil?",
    "¿Cuántos días tengo para devolver un producto?",
    "¿Cuál es la capital de Francia?",   # control: no debe responder
]

for pregunta in preguntas_prueba:
    print(f"\n{'='*70}")
    print(f"❓ {pregunta}")
    print(f"{'='*70}")
    respuesta = asistente.preguntar(pregunta)
    print(f"🤖 {respuesta}")


❓ ¿Cuánto tarda en procesarse un reembolso?
🤖 Una vez recibido e inspeccionado el producto, se procesará el reembolso en 5-7 días hábiles. Ten en cuenta que, según el método de pago, las tarjetas de crédito pueden tardar hasta 10 días adicionales según el banco.

(Fuente: Política de Devoluciones) [Fuente: Proceso de devolución]

❓ ¿Puedo pagar con Bizum?
🤖 Sí, es posible pagar con Bizum. Debe saber que es aplicable para pagos de hasta 2.500€ al día y está disponible solo para clientes con banco español. (Fuente: Métodos de Pago)

❓ ¿Cuánto cuesta el envío express?
🤖 El envío express cuesta 4,99€ y ofrece un plazo de entrega de 1-2 días hábiles [Fuente: Política de Envíos].

❓ ¿La garantía cubre si se me cae el móvil?
🤖 No, la garantía no cubre daños por golpes, caídas o líquidos.

(Fuentes: [Fuente: Garantías de Productos] [Fuente: Garantías de Productos])

❓ ¿Cuántos días tengo para devolver un producto?
🤖 Los clientes tienen 30 días naturales desde la fecha de recepción del pedido 

## 8. Chat interactivo

Bucle de chat. Escribe tu pregunta y pulsa Enter. Para terminar, escribe `salir`.

In [9]:
print("="*70)
print("💬 Chat con el asistente virtual de TechShop")
print("="*70)
print("Escribe tu pregunta (o 'salir' para terminar)\n")

while True:
    try:
        pregunta = input("Tú: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n👋 ¡Hasta luego!")
        break
    
    if not pregunta:
        continue
    if pregunta.lower() in ("salir", "exit", "quit", "fin"):
        print("\n👋 ¡Hasta luego!")
        break
    
    print()
    respuesta = asistente.preguntar(pregunta)
    print(f"🤖 Asistente: {respuesta}\n")

💬 Chat con el asistente virtual de TechShop
Escribe tu pregunta (o 'salir' para terminar)


👋 ¡Hasta luego!
